# Project 17 — BROKEN notebook (debugging exercise)

This notebook contains **seeded bugs** centred on the GP's signature pathology: a vague length-scale prior that makes $\ell$ and $\eta$ trade off. Run it, read the diagnostics (pair plot, divergences, $\hat R$), find each bug, and fix it. Answer key: `BROKEN_BUGS.md` (don't peek first).

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
RNG = 20240601

In [ ]:
from data.generate_data import generate
data = generate()
x, y = data['x'], data['y']

### Model — a vague length-scale prior and a starved sampler.

Two seeded bugs hide here. Watch the $(\ell,\eta)$ pair plot.

In [ ]:
# BUG 1: a vague, heavy-tailed length-scale prior (HalfFlat) lets ell wander
#         all the way out, re-opening the ell <-> eta trade-off.
# BUG 2: low target_accept (0.8) on this curved geometry -> divergences.
with pm.Model() as model:
    ell = pm.HalfFlat('ell')
    eta = pm.HalfNormal('eta', sigma=2.0)
    sigma = pm.HalfNormal('sigma', sigma=0.5)
    cov = eta**2 * pm.gp.cov.ExpQuad(input_dim=1, ls=ell)
    gp = pm.gp.Marginal(cov_func=cov)
    gp.marginal_likelihood('y_obs', X=x[:,None], y=y, sigma=sigma)
    idata = pm.sample(draws=400, tune=400, chains=2, target_accept=0.8,
                      random_seed=RNG, progressbar=False)

In [ ]:
print(az.summary(idata, var_names=['ell','eta','sigma']))
print('divergences:', int(idata.sample_stats['diverging'].sum()))

### The smoking gun — BUG 3: the pair plot reveals the ridge.

With a vague length-scale prior, $\ell$ and $\eta$ are correlated along a diagonal ridge (large $\ell$ + large $\eta$ explains the data as well as small $\ell$ + small $\eta$). The fix is the informative `InverseGamma` length-scale prior from `model.py`.

In [ ]:
az.plot_pair(idata, var_names=['ell','eta'], kind='scatter',
             scatter_kwargs={'alpha':0.3}); plt.tight_layout()